# Day 5 — Mini Project: LLM Explainer Dashboard

The capstone project of Week 1. This tool combines tokenization, attention, and next-token prediction into one interactive explainer.

In [1]:
!pip install transformers matplotlib --quiet

In [2]:
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch
import matplotlib.pyplot as plt

MODEL_NAME = "gpt2"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
attn_model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
gen_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

c:\Users\Nemochan\Desktop\ai-engineering-journey\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading gpt2...


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 2364.53it/s]


In [3]:
def explain_sentence(text: str, focus_word: str = None):
    print("\n" + "="*80)
    print(f"🤖 LLM EXPLAINER DASHBOARD")
    print("="*80)
    print(f"Input: \"{text}\"")
    print("-"*80)

    inputs = tokenizer(text, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # 1. Tokenization
    print("\n📋 1. TOKENIZATION")
    for i, t in enumerate(tokens):
        print(f"   {i:2d}: {t}")

    # 2. Attention Analysis
    if focus_word:
        target = f"Ġ{focus_word}" if f"Ġ{focus_word}" in tokens else focus_word
        if target in tokens:
            idx = tokens.index(target)
            with torch.no_grad():
                outputs = attn_model(**inputs)
            last_attn = outputs.attentions[-1][0].mean(dim=0)

            print(f"\n👁️ 2. ATTENTION from '{focus_word}'")
            print("   Token              Attention   Visual")
            print("   " + "-"*55)

            for i, tok in enumerate(tokens):
                if i > idx: break
                weight = last_attn[idx, i].item()
                bar = "█" * int(weight * 25)
                print(f"   {tok:15s} {weight:.4f}   {bar}")
        else:
            print(f"\n👁️ 2. ATTENTION: '{focus_word}' not found.")

    # 3. Next Token Prediction
    with torch.no_grad():
        outputs = gen_model(**inputs)
    next_logits = outputs.logits[0, -1]
    top5 = torch.topk(next_logits, 5)

    print("\n🔮 3. NEXT TOKEN PREDICTION")
    print("   Token               Score")
    print("   " + "-"*40)
    for score, idx in zip(top5.values, top5.indices):
        token = tokenizer.decode([idx])
        print(f"   {token:15s} {score.item():.3f}")

    print("\n" + "="*80 + "\n")

## Test the Explainer

In [4]:
# Test examples
explain_sentence("The firewall blocked the traffic because it was suspicious", focus_word="it")
explain_sentence("The data subject has the right to access", focus_word="subject")


🤖 LLM EXPLAINER DASHBOARD
Input: "The firewall blocked the traffic because it was suspicious"
--------------------------------------------------------------------------------

📋 1. TOKENIZATION
    0: The
    1: Ġfirewall
    2: Ġblocked
    3: Ġthe
    4: Ġtraffic
    5: Ġbecause
    6: Ġit
    7: Ġwas
    8: Ġsuspicious

👁️ 2. ATTENTION from 'it'
   Token              Attention   Visual
   -------------------------------------------------------
   The             0.6787   ████████████████
   Ġfirewall       0.0700   █
   Ġblocked        0.0568   █
   Ġthe            0.0315   
   Ġtraffic        0.0479   █
   Ġbecause        0.0601   █
   Ġit             0.0551   █

🔮 3. NEXT TOKEN PREDICTION
   Token               Score
   ----------------------------------------
   .               -76.458
   ,               -76.919
    of             -77.285
    and            -77.890
   ly              -78.369



🤖 LLM EXPLAINER DASHBOARD
Input: "The data subject has the right to access"
---------